# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of Analysis:** One row = one pseudonymized content item.

**Time Window (Professional Spec):**
- **Feature Window:** A trailing 90-day window of search performance ending at $T_0$ (the decision point).
- **Target Window:** The subsequent 30-day window starting at $T_0+1$.

For the professional release, this is implemented as a manual aggregation from `fact_content_daily_performance`. We use the feature window to build the model and the target window to define the label (e.g., did the page decline in the following month?).

In [ ]:
import os
import pandas as pd
import duckdb

# Configuration
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    print("⚠️ WARNING: HF_TOKEN not found in environment variables. Warehouse access will fail.")

# Setup DuckDB for Warehouse Access
def get_warehouse_data(t0_date):
    conn = duckdb.connect(database=':memory:')
    conn.execute("INSTALL httpfs; LOAD httpfs;")
    conn.execute("CREATE SECRET (TYPE HTTPFS, TOKEN ?)", [HF_TOKEN])
    
    # Professional Temporal Window SQL
    sql = f"""
    WITH 
    feature_window AS (
        SELECT 
            content_id,
            SUM(impressions) as impressions_90d,
            SUM(clicks) as clicks_90d,
            SUM(sessions) as sessions_90d,
            SUM(ai_sessions) as ai_sessions_90d,
            SUM(engaged_sessions) as engaged_sessions_90d,
            SUM(scroll_events) as scroll_events_90d,
            AVG(avg_position) as avg_position,
            COUNT(DISTINCT CASE WHEN impressions > 0 THEN date END) as days_with_impressions,
            COUNT(DISTINCT CASE WHEN sessions > 0 THEN date END) as days_with_sessions
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
        WHERE date BETWEEN '{t0_date}'::DATE - INTERVAL 90 DAYS AND '{t0_date}'::DATE
        GROUP BY content_id
    ),
    target_window AS (
        SELECT 
            content_id,
            SUM(impressions) as impressions_future
        FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance'
        WHERE date BETWEEN '{t0_date}'::DATE + INTERVAL 1 DAY AND '{t0_date}'::DATE + INTERVAL 30 DAYS
        GROUP BY content_id
    ),
    content_dims AS (
        SELECT 
            content_id, client_id, search_volume, competition, cpc, word_count, char_count, content_type, main_intent
        FROM 'hf://datasets/FlyRank/internship-warehouse/dim_content'
    )
    SELECT 
        d.*,
        f.*,
        CASE 
            WHEN t.impressions_future < (f.impressions_90d / 3.0) THEN 1 
            ELSE 0 
        END as is_declining_label
    FROM content_dims d
    JOIN feature_window f ON d.content_id = f.content_id
    JOIN target_window t ON d.content_id = t.content_id;
    """
    return conn.execute(sql).df()

# Use Training T0 as default for the data contract
T0_TRAIN = "2026-04-30"
try:
    df = get_warehouse_data(T0_TRAIN)
    print(f"Successfully loaded warehouse data for T0={T0_TRAIN}")
except Exception as e:
    print(f"Error loading warehouse data: {e}")
    # Fallback to starter CSV so the notebook remains runnable for those without HF_TOKEN
    print("Falling back to starter CSV...")
    csv_path = os.path.join(os.getcwd(), "..", "..", "data", "raw", "content_refresh_anonymized.csv")
    df = pd.read_csv(csv_path)
    # Mock the professional label for the starter CSV
    df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# Verify grain: one row per content_id
is_unique = df['content_id'].is_unique
print(f"Is content_id unique? {is_unique}")
print(f"Total content items: {len(df)}")
df[['content_id', 'client_id', 'impressions_90d']].head()

## 2. Fields: feature / label / context / excluded

I have sorted the fields based on their role in predicting the **Content Refresh Opportunity**.

**Features (Input signals):**
- `impressions_90d`: Volume signal (essential for filtering noise).
- `avg_position`: Primary driver of visibility.
- `content_type`, `main_intent`: Contextual baseline for engagement.
- `word_count`, `char_count`: Proxy for content depth.
- `content_age_days`: Freshness signal.
- `engagement_rate`, `scroll_rate`: On-page behavior signals.
- `ai_traffic_pct`: Signal of AI-driven discovery.

**Label (Target):**
- `is_declining_label`: A professional future-window target. 1 if the page's impressions in the following 30 days drop significantly below its trailing 90-day average.

**Context (Grouping/Metadata):**
- `content_id`: Primary key.
- `client_id`: Used for group-holdout validation.

**Excluded (Why):**
- `trend_direction`, `trend_pct`: These are current-window measurements. Including them would be a proxy for the answer, not a predictive signal for the future window.

In [ ]:
# Defining the buckets for the contract
buckets = {
    'features': ['impressions_90d', 'avg_position', 'content_type', 'main_intent', 'word_count', 'char_count', 'content_age_days', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'],
    'label': ['is_declining_label'],
    'context': ['content_id', 'client_id'],
    'excluded': ['trend_direction', 'trend_pct']
}

# Check if all fields exist in the dataset
all_fields = df.columns.tolist()
for bucket, fields in buckets.items():
    missing = [f for f in fields if f not in all_fields]
    print(f"{bucket}: {len(missing)} missing fields {missing}")

## 3. Verify it with queries (grain, counts, missing values, windows)

I will now verify the distributional health of the features and the validity of the proxy target.

In [ ]:
# 1. Verify Grain and Volume
print(f"Total rows: {len(df)}")
print(f"Unique Content IDs: {df['content_id'].nunique()}")

# 2. Verify Target (Future Decline)
visible_mask = (df['impressions_90d'] >= 500)
decline_mask = (df['is_declining_label'] == 1)
opportunity_mask = visible_mask & decline_mask

print(f"Pages with high visibility (>=500 imp): {visible_mask.sum()}")
print(f"Pages with high visibility AND predicted decline: {opportunity_mask.sum()}")
print(f"Opportunity Rate among visible pages: {opportunity_mask.sum()/visible_mask.sum():.2%}")

# 3. Check Missingness for key features
feature_cols = buckets['features']
# Filter for columns that actually exist in the current df
existing_features = [c for c in feature_cols if c in df.columns]
missing_stats = df[existing_features].isna().sum()
print("\nMissing values per feature:")
print(missing_stats[missing_stats > 0])

# 4. Check Position Distribution
if 'position_tier' in df.columns:
    print("\nPosition Distribution:")
    print(df['position_tier'].value_counts(normalize=True).sort_index())
else:
    print("\nPosition tier not available in this feature set.")

## 4. Data limits

Every dataset has boundaries. For this Refresh Opportunity analysis, the primary limits are:

1. **The Unbalanced Panel**: In the warehouse release, different clients have different history depths. Our temporal windowing handles this, but some clients may not have enough history to populate both the feature and target windows.
2. **The 'Zero-Click' Blindspot**: Our labels are based on impressions. However, a drop in impressions might be due to a change in search intent or a "zero-click" shift (answers appearing in the SERP), not necessarily a decline in content quality.
3. **GSC-Only Periods**: Early history for some clients lacks GA4 engagement data. I must use the `ga4_data_available` flag in the warehouse to avoid treating "no tracking" as "zero engagement."
4. **Observation vs Causality**: This is a predictive model. We can observe that certain signals are associated with future decline, but we cannot prove that a "refresh" will cause a recovery without a controlled experiment.

In [5]:
# Verification of GA4 availability (conceptually, as this is the starter CSV)
# In the warehouse, we would use: df[df['ga4_data_available'] == False]
# In the starter CSV, we check for NaN in engagement rates as a proxy for missing tracking
missing_engagement = df['engagement_rate'].isna().sum()
print(f"Rows with missing engagement_rate: {missing_engagement}")
print(f"Percentage missing: {missing_engagement/len(df):.2%}")

Rows with missing engagement_rate: 0
Percentage missing: 0.00%


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.